# AegisRAG: End-to-End Generative AI Governance Copilot

This executed notebook demonstrates source ingestion, hybrid retrieval, grounded generation, citation validation, AI risk screening, and evaluation.

In [1]:
from pathlib import Path
import json
from aegisrag import AegisRAG
from aegisrag.evaluation import evaluate

ROOT = Path.cwd()
rag = AegisRAG()
print('AegisRAG initialised successfully')

AegisRAG initialised successfully


## 1. Source and runtime summary

In [2]:
print(json.dumps(rag.source_summary(), indent=2))

{
  "documents": 16,
  "chunks": 29,
  "frameworks": {
    "EU AI Act": 10,
    "NIST AI RMF": 6
  },
  "embedding_backend": "lsa-tfidf",
  "llm_provider": "offline-evidence-generator"
}


## 2. Hybrid retrieval with component scores

In [3]:
query = 'What controls are expected for a high-risk AI system?'
hits = rag.retriever.search(query, top_k=5)
for hit in hits:
    print(f'{hit.rank}. {hit.chunk.reference:24} score={hit.score:.3f} '          f'lexical={hit.lexical_score:.3f} semantic={hit.semantic_score:.3f}')

1. Article 9                score=0.896 lexical=0.940 semantic=1.000
2. GOVERN function          score=0.671 lexical=1.000 semantic=0.457
3. Article 6 and Annex III  score=0.629 lexical=0.612 semantic=0.760
4. Regulation (EU) 2024/1689 score=0.534 lexical=0.827 semantic=0.338
5. Articles 11 and 12       score=0.504 lexical=0.643 semantic=0.484


## 3. Evidence-grounded answer with citations

In [4]:
question = 'What transparency duties may apply to a customer-service chatbot?'
answer = rag.ask(question)
print(answer.answer)
print(f'\nConfidence: {answer.confidence:.1%}')
print(f'Evidence quality: {answer.evidence_quality}')
for citation in answer.citations:
    print(f'[{citation.citation_id}] {citation.reference} -> {citation.source_url}')

Based on the retrieved official guidance, the main points are: - Providers of systems intended to interact directly with people must generally design them so that people are informed that they are interacting with AI, unless this is obvious to a reasonably well-informed, observant, and circumspect person, subject to conditions and exceptions in the law. [S1]
- Exact duties depend on provider or deployer role, content type, context, editorial responsibility, and legal exceptions. [S1]
- It uses a risk-based structure: certain practices are prohibited, high-risk systems are subject to lifecycle requirements, some systems have transparency duties, and many minimal-risk systems have no additional mandatory requirements under the Act. [S2]
- Other Union or national law may still apply. [S2]
- They should explain intended purpose, performance characteristics, limitations, input specifications, foreseeable circumstances that may create risk, human-oversight measures, maintenance, and logging 

## 4. AI-system risk screening and control gaps

In [5]:
profile_path = ROOT / 'data/sample_systems/recruitment_assistant.json'
profile = json.loads(profile_path.read_text())
assessment = rag.assess(profile)
print(f'System: {assessment.system_name}')
print(f'Result: {assessment.risk_level} ({assessment.confidence:.0%})')
print('\nTriggers:')
for item in assessment.triggers: print('-', item)
print('\nControl gaps:')
for item in assessment.control_gaps: print('-', item)

System: TalentMatch Candidate Ranking Assistant
Result: likely high-risk (78%)

Triggers:
- employment, worker management, or access to self-employment

Control gaps:
- Document a continuous AI risk-management process.
- Establish data quality, provenance, representativeness, and bias controls.
- Enable traceable event logging and retain records under an approved policy.
- Define and test accuracy, robustness, and cybersecurity thresholds.
- Operate post-deployment monitoring and incident escalation.


## 5. Reproducible offline evaluation

In [6]:
summary, rows = evaluate(rag)
print(json.dumps(summary, indent=2))

{
  "examples": 10,
  "retrieval_recall_at_k": 1.0,
  "mean_reciprocal_rank": 1.0,
  "answer_keyword_coverage": 0.675,
  "citation_coverage": 1.0,
  "average_confidence": 0.8244,
  "average_latency_ms": 0.98,
  "provider": "offline-evidence-generator",
  "embedding_backend": "lsa-tfidf"
}


## Result

The complete pipeline runs without an external LLM key. Hosted or local LLMs can be enabled through environment variables while keeping retrieval, guardrails, citations, risk-screening, evaluation, and audit components unchanged.